# Project Overview

This notebook generates a synthetic English text dataset for the High-Performance Similarity Search Engine project.

- **Target dataset size:** 12,000 documents
- **Language:** English
- **Model:** Llama 3.2 1B Instruct (4-bit NF4)
- **Minimum body length:** 50 tokens
- **Generation quality checks:** EOS completion, minimum length, sentence ending, unfinished-phrase heuristics
- **Recovery:** Saves after every batch and resumes missing document IDs
- **Output dataset:** `synthetic_documents_12000_1b_v2.csv`
- **Output prompts:** `prompts_12000_1b_v2.csv`




In [1]:
# Install the libraries required for dataset generation.
!pip -q install transformers accelerate bitsandbytes pandas


In [3]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Define project directories
PROJECT_DIR = Path("/content/drive/MyDrive/HPP_DatasetGeneration")
DATA_DIR = PROJECT_DIR / "data"

# Create directories if they do not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)


Mounted at /content/drive
Project directory: /content/drive/MyDrive/HPP_DatasetGeneration
Data directory: /content/drive/MyDrive/HPP_DatasetGeneration/data


In [4]:

import csv
import random

# Generate prompts for the text corpus
random.seed(42)

topics = [
    "artificial intelligence", "machine learning", "data science",
    "information retrieval", "search engines", "natural language processing",
    "computer vision", "robotics", "cybersecurity", "cloud computing",
    "database systems", "software engineering", "distributed systems",
    "computer networks", "operating systems", "data visualization",
    "recommendation systems", "human-computer interaction", "education technology",
    "healthcare technology", "renewable energy", "climate change",
    "space exploration", "biology", "psychology", "economics",
    "digital marketing", "financial technology", "smart cities",
    "Internet of Things", "blockchain", "mobile applications",
    "web development", "software testing", "project management",
    "digital privacy", "ethical technology", "scientific research",
    "online learning", "robotic automation", "big data",
    "data mining", "predictive analytics", "natural disasters",
    "sustainable agriculture", "urban planning", "public transportation",
    "digital libraries", "open-source software", "computer graphics",
    "game development", "virtual reality", "augmented reality",
    "quantum computing", "semiconductor technology", "battery technology",
    "electric vehicles", "supply chain management", "business analytics",
    "remote work", "digital communication", "social media",
    "media literacy", "language learning", "scientific communication",
    "public health", "nutrition science", "environmental conservation",
    "ocean research", "astronomy", "mathematics education",
    "critical thinking", "creative problem solving", "data ethics",
    "software architecture", "digital transformation", "knowledge management",
    "automated reasoning", "information security", "parallel computing"
]

tasks = [
    "Explain key concepts and practical applications of {topic}.",
    "Describe current challenges and possible solutions in {topic}.",
    "Discuss future research directions related to {topic}.",
    "Explain how {topic} can improve real-world decision-making.",
    "Compare common approaches used in {topic}.",
    "Describe an example project involving {topic}.",
    "Discuss the benefits and limitations of {topic}.",
    "Explain important design considerations for {topic}.",
    "Describe how beginners can learn the fundamentals of {topic}.",
    "Discuss how {topic} affects individuals and organizations.",
    "Explain common misconceptions about {topic}.",
    "Describe methods for evaluating systems related to {topic}.",
    "Discuss ethical considerations associated with {topic}.",
    "Explain how data can be used in {topic}.",
    "Describe practical ways to improve outcomes in {topic}.",
    "Discuss the relationship between {topic} and innovation.",
    "Explain common performance problems in {topic}.",
    "Describe how professionals collaborate on {topic}.",
    "Discuss emerging trends in {topic}.",
    "Explain why {topic} matters in modern society."
]

audiences = [
    "a university student",
    "a technical project team",
    "a general reader",
    "a professional audience"
]

styles = [
    "Use clear explanations and practical examples.",
    "Focus on useful details and real-world applications.",
    "Present the information in a logical, accessible way."
]


prompts = [
    {
        "doc_id": f"doc_{i:05d}",
        "topic": topic,
        "prompt": (
            f"Audience: {audience}. "
            f"Task: {task.format(topic=topic)} "
            f"{style} Write in English in about 70-90 words and 3-5 complete sentences. "
            "Use a coherent short paragraph, not a numbered or bulleted list. "
            "Include one specific useful detail or example. Finish with a complete, "
            "natural sentence. Avoid filler, repetition, unsupported claims, and "
            "repeating the instructions."
        )
    }
    for i, (topic, task, audience, style) in enumerate(
        (
            (topic, task, audience, style)
            for topic in topics
            for task in tasks
            for audience in audiences
            for style in styles
        ),
        start=1
    )
]

random.shuffle(prompts)
TARGET_DOCUMENTS = 12000
if len(prompts) < TARGET_DOCUMENTS:
    raise ValueError(
        f"Only {len(prompts)} unique prompt combinations are available; "
        f"{TARGET_DOCUMENTS} are required."
    )
prompts = prompts[:TARGET_DOCUMENTS]

PROMPTS_FILE = DATA_DIR / "prompts_12000_1b_v2.csv"

with PROMPTS_FILE.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f, fieldnames=["doc_id", "topic", "prompt"]
    )
    writer.writeheader()
    writer.writerows(prompts)

print("Number of prompts:", len(prompts))
print("Unique document IDs:", len({p["doc_id"] for p in prompts}))
print("Unique prompt texts:", len({p["prompt"] for p in prompts}))
print("Saved to:", PROMPTS_FILE)
print("\nExample prompt:")
print(prompts[0]["prompt"])


Number of prompts: 12000
Unique document IDs: 12000
Unique prompt texts: 12000
Saved to: /content/drive/MyDrive/HPP_DatasetGeneration/data/prompts_12000_1b_v2.csv

Example prompt:
Audience: a technical project team. Task: Describe practical ways to improve outcomes in machine learning. Use clear explanations and practical examples. Write in English in about 70-90 words and 3-5 complete sentences. Use a coherent short paragraph, not a numbered or bulleted list. Include one specific useful detail or example. Finish with a complete, natural sentence. Avoid filler, repetition, unsupported claims, and repeating the instructions.


In [9]:
# Rename prompt IDs without regenerating prompts
prompts_df = pd.read_csv(PROMPTS_FILE, dtype={"doc_id": str})

prompts_df = prompts_df.rename(
    columns={"doc_id": "prompt_id"}
)

prompts_df["prompt_id"] = prompts_df["prompt_id"].apply(
    lambda x: "prompt_" + x.removeprefix("doc_")
)

prompts_df.to_csv(PROMPTS_FILE, index=False)

print("Prompt IDs renamed successfully.")

Prompt IDs renamed successfully.


In [5]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()
print("Model loaded successfully.")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully.


In [7]:
import pandas as pd
import torch
from tqdm.auto import tqdm

# Load the 12,000 prompts for this version of the dataset
PROMPTS_FILE = DATA_DIR / "prompts_12000_1b_v2.csv"
OUTPUT_FILE = DATA_DIR / "synthetic_documents_12000_1b_v2.csv"
FAILURES_FILE = DATA_DIR / "failed_documents_12000_1b_v2.csv"

prompts_df = pd.read_csv(PROMPTS_FILE, dtype={"doc_id": str})

print("Prompts loaded:", len(prompts_df))
print("Unique prompt IDs:", prompts_df["doc_id"].nunique())
print("Output file:", OUTPUT_FILE)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Prompts loaded: 12000
Unique prompt IDs: 12000
Output file: /content/drive/MyDrive/HPP_DatasetGeneration/data/synthetic_documents_12000_1b_v2.csv
GPU available: True
GPU: Tesla T4


In [10]:

import pandas as pd
import torch
import time
from datetime import datetime
from tqdm.auto import tqdm

# Configuration
TARGET_DOCUMENTS = 12000
BATCH_SIZE = 40
MAX_NEW_TOKENS = 192
RETRY_MAX_NEW_TOKENS = 256
MAX_RETRIES = 3
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# File paths
PROMPTS_FILE = DATA_DIR / "prompts_12000_1b_v2.csv"
OUTPUT_FILE = DATA_DIR / "synthetic_documents_12000_1b_v2.csv"
FAILURES_FILE = DATA_DIR / "failed_documents_12000_1b_v2.csv"
MISSING_FILE = DATA_DIR / "missing_prompts_12000_1b_v2.csv"

SYSTEM_PROMPT = (
    "Write an informative English response of about 70-90 words in 3-5 complete "
    "sentences. Use one coherent paragraph, not a numbered or bulleted list. "
    "Include one useful detail or example and finish with a complete sentence."
)

COLUMNS = [
    "doc_id", "source_prompt_id", "topic", "text",
    "word_count", "token_count", "body_token_count",
    "model", "generated_at", "elapsed_seconds",
    "generation_attempts", "ended_with_eos"
]

# Load and validate prompts
prompts_df = pd.read_csv(
    PROMPTS_FILE,
    dtype={"prompt_id": str}
).fillna("")

required_columns = {"prompt_id", "topic", "prompt"}

if not required_columns.issubset(prompts_df.columns):
    raise ValueError(
        "Prompt file must contain prompt_id, topic, and prompt columns."
    )

if len(prompts_df) != TARGET_DOCUMENTS:
    raise ValueError(f"Expected exactly {TARGET_DOCUMENTS} prompts.")

if prompts_df["prompt_id"].nunique() != TARGET_DOCUMENTS:
    raise ValueError("Prompt IDs must be unique.")

# Prepare the model
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.eval()


def generate_batch(batch, max_tokens):
    # Format prompts for the chat model
    formatted = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": str(prompt)}
            ],
            tokenize=False,
            add_generation_prompt=True
        )
        for prompt in batch["prompt"]
    ]

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    # Generate responses
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[:, inputs["input_ids"].shape[1]:]
    results = []

    # Check whether each response reached EOS
    for token_ids in generated:
        ids = token_ids.tolist()
        ended_with_eos = tokenizer.eos_token_id in ids

        if ended_with_eos:
            ids = ids[:ids.index(tokenizer.eos_token_id)]

        text = tokenizer.decode(ids, skip_special_tokens=True).strip()

        results.append({
            "text": text,
            "word_count": len(text.split()),
            "token_count": len(ids),
            "body_token_count": len(
                tokenizer.encode(text, add_special_tokens=False)
            ),
            "ended_with_eos": ended_with_eos
        })

    return results


def save_result(item, result, attempts, elapsed, doc_id):
    # Preserve the original prompt ID
    return {
        "doc_id": doc_id,
        "source_prompt_id": str(item["prompt_id"]),
        "topic": str(item["topic"]),
        **result,
        "model": MODEL_NAME,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "elapsed_seconds": round(elapsed, 2),
        "generation_attempts": attempts
    }

print("Prompt validation completed.")
print(f"Prompts loaded: {len(prompts_df):,}")
print("Model and generation functions are ready.")


Prompt validation completed.
Prompts loaded: 12,000
Model and generation functions are ready.


In [11]:

# Resume from previously saved valid documents
if OUTPUT_FILE.exists():
    existing = pd.read_csv(
        OUTPUT_FILE,
        dtype={"doc_id": str, "source_prompt_id": str}
    ).fillna("")

    if not set(COLUMNS).issubset(existing.columns):
        raise ValueError(
            f"{OUTPUT_FILE.name} has an incompatible format. "
            "Rename it before starting."
        )

    # Keep one valid document per source prompt
    existing = existing.drop_duplicates(
        "source_prompt_id", keep="last"
    )

    eos_ok = existing["ended_with_eos"].astype(str).str.lower().isin(
        ["true", "1", "yes"]
    )
    existing = existing.loc[eos_ok, COLUMNS].copy()

    # Restore sequential document IDs
    existing = existing.sort_values("doc_id").reset_index(drop=True)
    existing["doc_id"] = [
        f"doc_{i:05d}" for i in range(1, len(existing) + 1)
    ]

    existing.to_csv(OUTPUT_FILE, index=False)
    completed_prompt_ids = set(existing["source_prompt_id"].astype(str))

else:
    existing = pd.DataFrame(columns=COLUMNS)
    existing.to_csv(OUTPUT_FILE, index=False)
    completed_prompt_ids = set()

# Randomize prompts that have not been completed
pending = prompts_df[
    ~prompts_df["prompt_id"].astype(str).isin(completed_prompt_ids)
].sample(frac=1).reset_index(drop=True)

next_doc_number = len(existing) + 1

print(f"Already completed: {len(existing):,}")
print(f"Remaining prompts: {len(pending):,}")
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "Not available"
)

started = time.time()
failed = []
newly_saved = 0

for start in tqdm(
    range(0, len(pending), BATCH_SIZE),
    desc="Generating"
):
    batch = pending.iloc[start:start + BATCH_SIZE].reset_index(drop=True)
    batch_started = time.time()

    results = generate_batch(batch, MAX_NEW_TOKENS)
    to_save = []

    for pos, (_, item) in enumerate(batch.iterrows()):
        result = results[pos]
        attempts = 1

        # Retry if EOS was not generated
        while not result["ended_with_eos"] and attempts <= MAX_RETRIES:
            attempts += 1
            result = generate_batch(
                pd.DataFrame([item]),
                RETRY_MAX_NEW_TOKENS
            )[0]

        if result["ended_with_eos"]:
            # Assign an ID to each successful document
            doc_id = f"doc_{next_doc_number:05d}"

            to_save.append(
                save_result(
                    item,
                    result,
                    attempts,
                    (time.time() - batch_started) / len(batch),
                    doc_id
                )
            )
            next_doc_number += 1

        else:
            failed.append({
                "source_prompt_id": str(item["prompt_id"]),
                "topic": str(item["topic"]),
                "attempts": attempts,
                "failure_reason": "EOS token not generated after retries",
                "last_text": result["text"]
            })

    # Save each batch immediately
    if to_save:
        pd.DataFrame(to_save, columns=COLUMNS).to_csv(
            OUTPUT_FILE,
            mode="a",
            header=False,
            index=False
        )
        newly_saved += len(to_save)

    if (start // BATCH_SIZE + 1) % 10 == 0 or start + BATCH_SIZE >= len(pending):
        print(
            f"Saved this run: {newly_saved:,} | "
            f"Failed after retries: {len(failed)} | "
            f"Elapsed: {(time.time() - started) / 60:.1f} min"
        )

# Save failed prompts
if failed:
    pd.DataFrame(failed).to_csv(FAILURES_FILE, index=False)
    print("Failed prompts saved to:", FAILURES_FILE)
elif FAILURES_FILE.exists():
    FAILURES_FILE.unlink()

print("Generation cell completed.")
print("Output file:", OUTPUT_FILE)


Already completed: 0
Remaining prompts: 12,000
GPU: Tesla T4


Generating:   0%|          | 0/300 [00:00<?, ?it/s]

Saved this run: 400 | Failed after retries: 0 | Elapsed: 2.4 min
Saved this run: 800 | Failed after retries: 0 | Elapsed: 5.0 min
Saved this run: 1,200 | Failed after retries: 0 | Elapsed: 7.4 min
Saved this run: 1,600 | Failed after retries: 0 | Elapsed: 9.9 min
Saved this run: 2,000 | Failed after retries: 0 | Elapsed: 12.3 min
Saved this run: 2,400 | Failed after retries: 0 | Elapsed: 14.7 min
Saved this run: 2,800 | Failed after retries: 0 | Elapsed: 17.0 min
Saved this run: 3,200 | Failed after retries: 0 | Elapsed: 19.4 min
Saved this run: 3,600 | Failed after retries: 0 | Elapsed: 21.9 min
Saved this run: 4,000 | Failed after retries: 0 | Elapsed: 24.3 min
Saved this run: 4,400 | Failed after retries: 0 | Elapsed: 26.7 min
Saved this run: 4,800 | Failed after retries: 0 | Elapsed: 29.1 min
Saved this run: 5,200 | Failed after retries: 0 | Elapsed: 31.6 min
Saved this run: 5,600 | Failed after retries: 0 | Elapsed: 33.9 min
Saved this run: 6,000 | Failed after retries: 0 | Elapse

In [12]:

import pandas as pd

# Load the dataset
file_path = DATA_DIR / "synthetic_documents_12000_1b_v2.csv"
df = pd.read_csv(file_path)

# Check dataset requirements
print("Documents:", len(df))
print("Unique document IDs:", df["doc_id"].nunique())
print("Average body tokens:", round(df["body_token_count"].mean(), 2))
print("Missing texts:", df["text"].isna().sum())
print("Empty texts:", df["text"].fillna("").str.strip().eq("").sum())
print("Duplicate texts:", df["text"].duplicated().sum())
print(
    "Documents with EOS:",
    df["ended_with_eos"].astype(str).str.lower().isin(["true", "1", "yes"]).sum()
)
print("\nColumns:", df.columns.tolist())


Documents: 12000
Unique document IDs: 12000
Average body tokens: 103.47
Missing texts: 0
Empty texts: 0
Duplicate texts: 0
Documents with EOS: 12000

Columns: ['doc_id', 'source_prompt_id', 'topic', 'text', 'word_count', 'token_count', 'body_token_count', 'model', 'generated_at', 'elapsed_seconds', 'generation_attempts', 'ended_with_eos']


In [14]:

import pandas as pd
import textwrap

# Load the dataset
file_path = DATA_DIR / "synthetic_documents_12000_1b_v2.csv"
df = pd.read_csv(file_path)

# Select 5 random documents
sample = df.sample(n=5)

for _, row in sample.iterrows():
    print("=" * 80)
    print(f"Document ID: {row['doc_id']}")
    print(f"Source Prompt ID: {row['source_prompt_id']}")
    print(f"Topic: {row['topic']}")
    print(f"Word count: {row['word_count']}")
    print(f"Body tokens: {row['body_token_count']}")
    print(f"EOS: {row['ended_with_eos']}")
    print("\nText:")

    # Wrap text at 80 characters
    print(textwrap.fill(str(row["text"]), width=80))
    print()


Document ID: doc_01050
Source Prompt ID: prompt_05188
Topic: climate change
Word count: 91
Body tokens: 104
EOS: True

Text:
As the world grapples with the challenges of climate change, it is essential to
consider the ethical implications of our actions. One crucial aspect is the role
of technology in mitigating climate change. For instance, the development of
renewable energy sources such as solar and wind power can help reduce carbon
emissions and promote a cleaner energy mix. Moreover, investing in energy
efficiency measures can also contribute to a more sustainable future. By
adopting a holistic approach to sustainability, we can work towards a more
environmentally conscious and responsible technology-driven future.

Document ID: doc_09765
Source Prompt ID: prompt_06037
Topic: economics
Word count: 76
Body tokens: 88
EOS: True

Text:
The application of economics to real-world decision-making can significantly
improve outcomes. Economics provides a systematic approach to understandi